# Lab 10 · Final project and timed practice

**Part A** (about 2 h 30 min): build a small lakehouse with Delta Lake from messy raw files: bronze (raw), silver (clean) and gold (reports), then load a new month incrementally.<br>
**Part B** (about 90 min): 10 timed problems in the format of coding interviews: write a function `etl(...)` that returns a DataFrame.

This lab uses everything from Labs 01 to 09. In the Theory part of Day 10, do the mock checkpoint over all checklist sections.

## How to work
* Run the **Setup** cell first. `F` is `pyspark.sql.functions`, `Window` is `pyspark.sql.Window`, `DeltaTable` is imported.
* **Part A**: the milestones build on each other: do them in order, and keep your functions, you need them again. Your output must look like the **expected output**.
* **Part B**: set a timer. Each problem shows its level and a target time. Write `etl(...)` from scratch, run `etl(...).show()`, and compare with the expected output. There are no hints: that's the point.
* The answers are collapsed at the very end of the notebook. Open one only after you have tried.

## Documentation
* [Medallion architecture](https://www.databricks.com/glossary/medallion-architecture): bronze, silver and gold
* [Delta Lake: update, delete and merge](https://docs.delta.io/latest/delta-update.html)
* [Delta Lake: time travel and schema enforcement](https://docs.delta.io/latest/delta-batch.html)
* [Delta Lake: OPTIMIZE](https://docs.delta.io/latest/optimizations-oss.html) and [history](https://docs.delta.io/latest/delta-utility.html)
* [DataFrame API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html) and [Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)
* [Window](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/window.html)

## The data
Three small tables of an online shop. The setup cell creates them, and you can read every row in it.

* **customers**: *customer_id*, *name*, *email*, *city*, *signup_date* (the day the customer registered)
* **products**: *product_id*, *product_name*, *category*, *price* (dollars for one item)
* **orders**: *order_id*, *customer_id*, *product_id*, *quantity*, *order_date*, *status* (`delivered`, `shipped` or `cancelled`)

Good to know:
* order 1011 is a guest checkout: it has no *customer_id*
* customer 6 has no email and no city
* customer 7 has never ordered, and product 107 has never been sold
* order 1016 is for product 108, which is no longer in *products*

**Part A** starts from three raw CSV files in `data/shop/raw/` (the setup cell writes them): `orders_2024-01.csv`, `orders_2024-02.csv` and `orders_2024-03.csv`, with the columns of *orders* as text. They contain mistakes on purpose: a row loaded twice, a quantity of 0 or below, a product that doesn't exist, an impossible date, and in March a status change of an older order.

**Part B** uses *customers*, *products* and *orders*.

## Setup
Run this cell once. It starts Spark with Delta Lake, saves the three shop tables as Parquet files in `data/shop/`, and writes the raw monthly files to `data/shop/raw/`. The first start may print a list of packages: that is Spark loading the Delta library.

In [ ]:
# Setup: starts Spark and creates the shop tables. Run it once.
import logging, os, shutil, sys, time
from datetime import date
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this notebook's Python
from pyspark.sql import SparkSession, Window, functions as F
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

builder = (SparkSession.builder
           .master("local[4]")   # 4 tasks at a time on your machine
           .appName("lab10")
           .config("spark.sql.adaptive.enabled", "true")        # AQE on, as in Spark's default
           .config("spark.sql.session.timeZone", "UTC")
           .config("spark.sql.shuffle.partitions", "4")          # small data: 4 are enough
           .config("spark.ui.showConsoleProgress", "false")
           .config("spark.log.level", "FATAL")                  # hide Spark's log messages
           # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this
           .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
spark = configure_spark_with_delta_pip(builder).getOrCreate()
for name in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
    logging.getLogger(name).setLevel(logging.CRITICAL)        # errors still show, without an extra JSON log

customers_rows = [
    # customer_id, name, email, city, signup_date
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Berlin",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     None,               None,       date(2024, 3, 10)),
    (7, "Grace Lee",        "grace@web.de",     "Berlin",   date(2024, 3, 15)),
]
products_rows = [
    # product_id, product_name, category, price
    (101, "Laptop",       "Electronics", 1200.0),
    (102, "Headphones",   "Electronics",  150.0),
    (103, "Coffee Maker", "Kitchen",       85.0),
    (104, "Blender",      "Kitchen",       60.0),
    (105, "Novel",        "Books",         15.0),
    (106, "Cookbook",     "Books",         25.0),
    (107, "Desk Lamp",    "Home",          40.0),
]
orders_rows = [
    # order_id, customer_id, product_id, quantity, order_date, status
    (1001, 1,    101, 1, date(2024, 1, 10), "delivered"),
    (1002, 1,    105, 3, date(2024, 1, 10), "delivered"),
    (1003, 2,    102, 2, date(2024, 1, 25), "delivered"),
    (1004, 3,    103, 1, date(2024, 2, 3),  "cancelled"),
    (1005, 2,    106, 1, date(2024, 2, 5),  "delivered"),
    (1006, 4,    104, 2, date(2024, 2, 15), "shipped"),
    (1007, 1,    102, 1, date(2024, 2, 20), "delivered"),
    (1008, 5,    101, 1, date(2024, 3, 2),  "shipped"),
    (1009, 3,    105, 4, date(2024, 3, 5),  "delivered"),
    (1010, 4,    106, 2, date(2024, 3, 8),  "cancelled"),
    (1011, None, 104, 1, date(2024, 3, 9),  "delivered"),
    (1012, 6,    103, 2, date(2024, 3, 12), "shipped"),
    (1013, 2,    101, 1, date(2024, 3, 18), "delivered"),
    (1014, 5,    105, 5, date(2024, 3, 20), "delivered"),
    (1015, 1,    103, 1, date(2024, 3, 25), "shipped"),
    (1016, 3,    108, 1, date(2024, 3, 28), "delivered"),
]

# Write the tables as Parquet files and read them back, the way real data arrives.
# data/shop belongs to this lab: it is rebuilt every time you run this cell.
shutil.rmtree("data/shop", ignore_errors=True)
for table, rows, schema in [
    ("customers", customers_rows, "customer_id INT, name STRING, email STRING, city STRING, signup_date DATE"),
    ("products",  products_rows,  "product_id INT, product_name STRING, category STRING, price DOUBLE"),
    ("orders",    orders_rows,    "order_id INT, customer_id INT, product_id INT, quantity INT, order_date DATE, status STRING"),
]:
    spark.createDataFrame(rows, schema).coalesce(1).write.parquet(f"data/shop/{table}")

customers = spark.read.parquet("data/shop/customers")
products = spark.read.parquet("data/shop/products")
orders = spark.read.parquet("data/shop/orders")

# Part A: the raw files, as the shop's system exports them every month (text, with mistakes)
HEADER = "order_id,customer_id,product_id,quantity,order_date,status"
RAW_FILES = {
    "orders_2024-01.csv": [
        "1001,1,101,1,2024-01-10,delivered",
        "1002,1,105,3,2024-01-10,delivered",
        "1002,1,105,3,2024-01-10,delivered",
        "1003,2,102,2,2024-01-25,delivered",
        "1018,4,104,0,2024-01-28,delivered",
    ],
    "orders_2024-02.csv": [
        "1004,3,103,1,2024-02-03,cancelled",
        "1005,2,106,1,2024-02-05,delivered",
        "1006,4,104,2,2024-02-15,shipped",
        "1007,1,102,1,2024-02-20,delivered",
        "1019,5,999,1,2024-02-21,shipped",
        "1020,6,103,1,2024-02-30,delivered",
    ],
    "orders_2024-03.csv": [
        "1003,2,102,2,2024-01-25,returned",
        "1008,5,101,1,2024-03-02,shipped",
        "1009,3,105,4,2024-03-05,delivered",
        "1010,4,106,2,2024-03-08,cancelled",
        "1011,,104,1,2024-03-09,delivered",
        "1012,6,103,2,2024-03-12,shipped",
        "1013,2,101,1,2024-03-18,delivered",
        "1014,5,105,5,2024-03-20,delivered",
        "1015,1,103,1,2024-03-25,shipped",
        "1016,3,108,1,2024-03-28,delivered",
        "1021,2,105,-2,2024-03-29,delivered",
    ],
}
os.makedirs("data/shop/raw", exist_ok=True)
for file_name, lines in RAW_FILES.items():
    with open(f"data/shop/raw/{file_name}", "w") as f:
        f.write("\n".join([HEADER] + lines) + "\n")

print("Spark", spark.version, "| Spark UI:", spark.sparkContext.uiWebUrl)

Look at the tables and one raw file:

In [ ]:
customers.show()
products.show()
orders.show()
print(open("data/shop/raw/orders_2024-02.csv").read())

## Part A · Project: a small shop lakehouse
Six milestones. Each table is a Delta table under `data/shop/lake/`. **Bronze** keeps the raw data exactly as it arrived, **silver** holds clean, typed, deduplicated rows (rejected rows go to a **quarantine** table with a reason), and **gold** holds the reports the business reads.

**P1. Bronze. Write `ingest(month)`: it appends one raw file (`data/shop/raw/orders_<month>.csv`) to the Delta table `data/shop/lake/bronze`, with every column as text plus `source_file` (the file name). Loading the same file twice must change nothing. Load January, February and January again, then count the rows per source_file.**

Expected output:

```
+------------------+-----+
|source_file       |count|
+------------------+-----+
|orders_2024-01.csv|5    |
|orders_2024-02.csv|6    |
+------------------+-----+
```

**P2. Silver. Write `check(bronze_rows)`: it removes exact duplicates, types the columns, and adds `reason` (null for good rows), testing in this order: `invalid date`, `bad quantity` (not a number, or 0 or below), `unknown product`. Write the good rows (without reason and source_file) to `data/shop/lake/silver` and the rejected rows to `data/shop/lake/quarantine`. Print the silver row count and the quarantine rows per reason.**

Expected output:

```
silver rows: 7
+---------------+-----+
|         reason|count|
+---------------+-----+
|   bad quantity|    1|
|   invalid date|    1|
|unknown product|    1|
+---------------+-----+
```

**P3. Gold. Write `build_gold(silver_rows)`: the revenue (quantity × price) of delivered orders per month (as text, like `2024-01`) and category. Save it to `data/shop/lake/gold` and show it, sorted by month and category.**

Expected output:

```
+-------+-----------+-------+
|  month|   category|revenue|
+-------+-----------+-------+
|2024-01|      Books|   45.0|
|2024-01|Electronics| 1500.0|
|2024-02|      Books|   25.0|
|2024-02|Electronics|  150.0|
+-------+-----------+-------+
```

**P4. March arrives. Ingest it, check it, add its rejected rows to quarantine, and MERGE its good rows into silver (update orders that exist, insert new ones). Print the silver row count and the status of order 1003.**

Expected output:

```
silver rows: 15
+--------+--------+
|order_id|  status|
+--------+--------+
|    1003|returned|
+--------+--------+
```

**P5. Rebuild gold from silver and compare it with gold's version 0 (time travel): show the month and category rows whose revenue changed or is new, with revenue_before and revenue_after.**

Expected output:

```
+-------+-----------+--------------+-------------+
|  month|   category|revenue_before|revenue_after|
+-------+-----------+--------------+-------------+
|2024-01|Electronics|        1500.0|       1200.0|
|2024-03|      Books|          NULL|        135.0|
|2024-03|Electronics|          NULL|       1200.0|
|2024-03|    Kitchen|          NULL|         60.0|
+-------+-----------+--------------+-------------+
```

**P6. The quarantine table gets a small append with every load. Compact it with `OPTIMIZE`, print its number of files before and after, and show its history: version and operation, in version order.**

Expected output:

```
2 files before
1 file after
+-------+---------+
|version|operation|
+-------+---------+
|      0|    WRITE|
|      1|    WRITE|
|      2| OPTIMIZE|
+-------+---------+
```

**Done with Part A?** Write a few lines for your team in a markdown cell: how many rows each rule rejected, why MERGE (and not append) loads silver, and what you would schedule daily (ingest, check, merge, gold, OPTIMIZE) and weekly (VACUUM).

## Part B · Timed practice
10 problems in the format of coding interviews. Each problem gives the tables your function gets; write `etl(...)` so that it **returns** a DataFrame, then run `etl(...).show()`. Set a timer: easy 5 minutes, medium 10, hard 15. Columns and row order must match the expected output.

**T1. (easy, 5 min) `etl(orders)`: the number of orders per day of the week, Monday first; days without orders are left out. Columns: weekday (like `Mon`) and orders.**

Expected output:

```
+-------+------+
|weekday|orders|
+-------+------+
|    Mon|     3|
|    Tue|     3|
|    Wed|     3|
|    Thu|     3|
|    Fri|     1|
|    Sat|     3|
+-------+------+
```

**T2. (easy, 5 min) `etl(customers)`: the number of customers per email domain (the part after `@`), most first, ties A to Z. Customers without an email are left out.**

Expected output:

```
+----------+---------+
|    domain|customers|
+----------+---------+
|   mail.uz|        2|
|    web.de|        2|
| correo.es|        1|
|post.co.uk|        1|
+----------+---------+
```

**T3. (easy, 5 min) `etl(orders, products)`: the products that were ordered but never delivered. Columns: product_id, product_name.**

Expected output:

```
+----------+------------+
|product_id|product_name|
+----------+------------+
|       103|Coffee Maker|
+----------+------------+
```

**T4. (medium, 10 min) `etl(orders, customers)`: in each city, the customer with the most orders (ties: the smaller customer_id). Customers without a city are left out. Columns: city, name, orders.**

Expected output:

```
+--------+-------------+------+
|    city|         name|orders|
+--------+-------------+------+
|  Berlin|Bella Schmidt|     3|
|  London|  Emma Wilson|     2|
|  Madrid|  Carlos Ruiz|     3|
|Tashkent| Aziz Karimov|     4|
+--------+-------------+------+
```

**T5. (medium, 10 min) `etl(orders, products)`: the delivered revenue (quantity × price) per month and its growth against the previous month in percent (1 decimal, empty for the first month). Columns: month, revenue, growth_pct.**

Expected output:

```
+-----+-------+----------+
|month|revenue|growth_pct|
+-----+-------+----------+
|    1| 1545.0|      NULL|
|    2|  175.0|     -88.7|
|    3| 1395.0|     697.1|
+-----+-------+----------+
```

**T6. (medium, 10 min) `etl(orders)`: the customers whose first order (by date, then order_id) was delivered but whose latest order was not. Columns: customer_id, first_status, last_status.**

Expected output:

```
+-----------+------------+-----------+
|customer_id|first_status|last_status|
+-----------+------------+-----------+
|          1|   delivered|    shipped|
+-----------+------------+-----------+
```

**T7. (medium, 10 min) `etl(orders, products)`: in each category, the customer who received the most delivered items (ties: the smaller customer_id; guest orders don't count). Columns: category, customer_id, items.**

Expected output:

```
+-----------+-----------+-----+
|   category|customer_id|items|
+-----------+-----------+-----+
|      Books|          5|    5|
|Electronics|          2|    3|
+-----------+-----------+-----+
```

**T8. (hard, 15 min) `etl(orders)`: for each customer with at least 2 orders, the longest gap in days between two consecutive orders. Columns: customer_id, longest_gap_days.**

Expected output:

```
+-----------+----------------+
|customer_id|longest_gap_days|
+-----------+----------------+
|          1|              41|
|          2|              42|
|          3|              31|
|          4|              22|
|          5|              18|
+-----------+----------------+
```

**T9. (hard, 15 min) `etl(orders)`: for each customer, the longest run of consecutive months (1, 2, 3) in which they ordered. Columns: customer_id, longest_streak.**

Expected output:

```
+-----------+--------------+
|customer_id|longest_streak|
+-----------+--------------+
|          1|             3|
|          2|             3|
|          3|             2|
|          4|             2|
|          5|             1|
|          6|             1|
+-----------+--------------+
```

**T10. (hard, 15 min) `etl(orders, products)`: rank the customers by delivered revenue (1 = highest; guests don't count) and show how much less each one spent than the customer ranked just above. Columns: customer_id, revenue, rank, gap_to_next_better.**

Expected output:

```
+-----------+-------+----+------------------+
|customer_id|revenue|rank|gap_to_next_better|
+-----------+-------+----+------------------+
|          2| 1525.0|   1|              NULL|
|          1| 1395.0|   2|             130.0|
|          5|   75.0|   3|            1320.0|
|          3|   60.0|   4|              15.0|
+-----------+-------+----+------------------+
```

---
**Done?** In the Theory part of Day 10, do the mock checkpoint: answer questions from every section aloud.

## Answers
Try each question yourself first. Click a question to open its answer; Part A answers also show the output of the reference run.

<details>
<summary><b>P1</b> &nbsp;Bronze. Write ingest(month): it appends one raw file (data/shop/raw/orders_&lt;month&gt;.csv) to the Delta table ...</summary>

```python
BRONZE = "data/shop/lake/bronze"

def ingest(month):
    """Appends one raw monthly file to bronze, every column as text, plus source_file. Does nothing if already loaded."""
    file_name = f"orders_{month}.csv"
    if DeltaTable.isDeltaTable(spark, BRONZE):
        loaded = spark.read.format("delta").load(BRONZE).filter(F.col("source_file") == file_name)
        if loaded.limit(1).count() > 0:
            return
    (spark.read.option("header", True).csv(f"data/shop/raw/{file_name}")
          .withColumn("source_file", F.col("_metadata.file_name"))
          .write.format("delta").mode("append").save(BRONZE))

ingest("2024-01")
ingest("2024-02")
ingest("2024-01")          # loading a file twice must change nothing
(spark.read.format("delta").load(BRONZE)
      .groupBy("source_file").count()
      .orderBy("source_file")
      .show(truncate=False))
```

Idempotent loading: rerunning a load (after a failure, or by mistake) must not duplicate data. Checking `source_file` before appending is the simplest way; `_metadata.file_name` gives the file of every row.

</details>

<details>
<summary><b>P2</b> &nbsp;Silver. Write check(bronze_rows): it removes exact duplicates, types the columns, and adds reason (null for ...</summary>

```python
SILVER, QUARANTINE = "data/shop/lake/silver", "data/shop/lake/quarantine"

def check(bronze_rows):
    """Removes exact duplicates, types the columns, and adds `reason`: why a row is rejected (null for good rows)."""
    typed = (bronze_rows.dropDuplicates(["order_id", "customer_id", "product_id", "quantity", "order_date", "status"])
                        .select(F.col("order_id").cast("int").alias("order_id"),
                                F.col("customer_id").cast("int").alias("customer_id"),
                                F.col("product_id").cast("int").alias("product_id"),
                                F.col("quantity").try_cast("int").alias("quantity"),
                                F.try_to_date("order_date", "yyyy-MM-dd").alias("order_date"),
                                "status", "source_file"))
    known = products.select("product_id", F.lit(True).alias("known_product"))
    return (typed.join(known, "product_id", "left")
                 .withColumn("reason", F.when(F.col("order_date").isNull(), "invalid date")
                                        .when(F.col("quantity").isNull() | (F.col("quantity") <= 0), "bad quantity")
                                        .when(F.col("known_product").isNull(), "unknown product"))
                 .select("order_id", "customer_id", "product_id", "quantity", "order_date", "status", "source_file", "reason"))

checked = check(spark.read.format("delta").load(BRONZE))
(checked.filter(F.col("reason").isNull()).drop("reason", "source_file")
        .write.format("delta").mode("overwrite").save(SILVER))
checked.filter(F.col("reason").isNotNull()).write.format("delta").mode("overwrite").save(QUARANTINE)

print("silver rows:", spark.read.format("delta").load(SILVER).count())
spark.read.format("delta").load(QUARANTINE).groupBy("reason").count().orderBy("reason").show()
```

`try_cast` and `try_to_date` turn bad values into nulls instead of failing the job. Keeping rejected rows in a quarantine table with a reason means nothing disappears silently.

</details>

<details>
<summary><b>P3</b> &nbsp;Gold. Write build_gold(silver_rows): the revenue (quantity × price) of delivered orders per month (as text, ...</summary>

```python
GOLD = "data/shop/lake/gold"

def build_gold(silver_rows):
    """Revenue of delivered orders per month and category."""
    return (silver_rows.filter(F.col("status") == "delivered")
                       .join(products, "product_id")
                       .groupBy(F.date_format("order_date", "yyyy-MM").alias("month"), "category")
                       .agg(F.sum(F.col("quantity") * F.col("price")).alias("revenue")))

build_gold(spark.read.format("delta").load(SILVER)).write.format("delta").mode("overwrite").save(GOLD)
spark.read.format("delta").load(GOLD).orderBy("month", "category").show()
```

</details>

<details>
<summary><b>P4</b> &nbsp;March arrives. Ingest it, check it, add its rejected rows to quarantine, and MERGE its good rows into silver ...</summary>

```python
ingest("2024-03")
march = check(spark.read.format("delta").load(BRONZE).filter(F.col("source_file") == "orders_2024-03.csv"))
march.filter(F.col("reason").isNotNull()).write.format("delta").mode("append").save(QUARANTINE)
good = march.filter(F.col("reason").isNull()).drop("reason", "source_file")
(DeltaTable.forPath(spark, SILVER).alias("t")
     .merge(good.alias("s"), "t.order_id = s.order_id")
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())

silver = spark.read.format("delta").load(SILVER)
print("silver rows:", silver.count())
silver.filter(F.col("order_id") == 1003).select("order_id", "status").show()
```

Order 1003 was in January's file as delivered; March's file says returned. MERGE updates it in place, so silver always holds the latest state of each order.

</details>

<details>
<summary><b>P5</b> &nbsp;Rebuild gold from silver and compare it with gold's version 0 (time travel): show the month and category rows ...</summary>

```python
build_gold(spark.read.format("delta").load(SILVER)).write.format("delta").mode("overwrite").save(GOLD)
before = spark.read.format("delta").option("versionAsOf", 0).load(GOLD).alias("before")
after = spark.read.format("delta").load(GOLD).alias("after")
(after.join(before, ["month", "category"], "full")
      .select("month", "category",
              F.col("before.revenue").alias("revenue_before"), F.col("after.revenue").alias("revenue_after"))
      .filter(~F.col("revenue_before").eqNullSafe(F.col("revenue_after")))
      .orderBy("month", "category")
      .show())
```

January's electronics revenue fell because order 1003 is now returned. A full rebuild of gold is fine for small tables; for big ones you would MERGE only the months that changed (read silver's change data feed, Lab 09).

</details>

<details>
<summary><b>P6</b> &nbsp;The quarantine table gets a small append with every load. Compact it with OPTIMIZE, print its number of files ...</summary>

```python
quarantine_table = DeltaTable.forPath(spark, QUARANTINE)
print(quarantine_table.detail().first()["numFiles"], "files before")
quarantine_table.optimize().executeCompaction()
print(quarantine_table.detail().first()["numFiles"], "file after")
quarantine_table.history().select("version", "operation").orderBy("version").show()
```

Every load adds small files; `OPTIMIZE` merges them so reads stay fast. Every step is a commit in the log (the first write, March's append, the OPTIMIZE): that history is your audit trail, and time travel can return to any of these versions until `VACUUM` removes their files.

</details>

<details>
<summary><b>T1</b> &nbsp;(easy, 5 min) etl(orders): the number of orders per day of the week, Monday first; days without orders are ...</summary>

```python
def etl(orders):
    return (orders.groupBy(F.weekday("order_date").alias("day_number"), F.date_format("order_date", "E").alias("weekday"))
                  .agg(F.count("*").alias("orders"))
                  .orderBy("day_number")
                  .select("weekday", "orders"))

etl(orders).show()
```

</details>

<details>
<summary><b>T2</b> &nbsp;(easy, 5 min) etl(customers): the number of customers per email domain (the part after @), most first, ties A ...</summary>

```python
def etl(customers):
    return (customers.filter(F.col("email").isNotNull())
                     .groupBy(F.substring_index("email", "@", -1).alias("domain"))
                     .agg(F.count("*").alias("customers"))
                     .orderBy(F.desc("customers"), "domain"))

etl(customers).show()
```

</details>

<details>
<summary><b>T3</b> &nbsp;(easy, 5 min) etl(orders, products): the products that were ordered but never delivered. Columns: product_id, ...</summary>

```python
def etl(orders, products):
    ordered = orders.select("product_id").distinct()
    delivered = orders.filter(F.col("status") == "delivered").select("product_id")
    return (ordered.join(delivered, "product_id", "left_anti")
                   .join(products, "product_id")
                   .select("product_id", "product_name")
                   .orderBy("product_id"))

etl(orders, products).show()
```

</details>

<details>
<summary><b>T4</b> &nbsp;(medium, 10 min) etl(orders, customers): in each city, the customer with the most orders (ties: the smaller ...</summary>

```python
def etl(orders, customers):
    counts = orders.groupBy("customer_id").agg(F.count("*").alias("orders"))
    w = Window.partitionBy("city").orderBy(F.desc("orders"), "customer_id")
    return (customers.filter(F.col("city").isNotNull())
                     .join(counts, "customer_id")
                     .withColumn("rn", F.row_number().over(w))
                     .filter(F.col("rn") == 1)
                     .select("city", "name", "orders")
                     .orderBy("city"))

etl(orders, customers).show()
```

</details>

<details>
<summary><b>T5</b> &nbsp;(medium, 10 min) etl(orders, products): the delivered revenue (quantity × price) per month and its growth ...</summary>

```python
def etl(orders, products):
    monthly = (orders.filter(F.col("status") == "delivered")
                     .join(products, "product_id")
                     .groupBy(F.month("order_date").alias("month"))
                     .agg(F.sum(F.col("quantity") * F.col("price")).alias("revenue")))
    previous = F.lag("revenue").over(Window.orderBy("month"))
    return (monthly.withColumn("growth_pct", F.round((F.col("revenue") - previous) / previous * 100, 1))
                   .orderBy("month"))

etl(orders, products).show()
```

</details>

<details>
<summary><b>T6</b> &nbsp;(medium, 10 min) etl(orders): the customers whose first order (by date, then order_id) was delivered but ...</summary>

```python
def etl(orders):
    by_time = F.struct("order_date", "order_id")
    return (orders.filter(F.col("customer_id").isNotNull())
                  .groupBy("customer_id")
                  .agg(F.min_by("status", by_time).alias("first_status"), F.max_by("status", by_time).alias("last_status"))
                  .filter((F.col("first_status") == "delivered") & (F.col("last_status") != "delivered"))
                  .orderBy("customer_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>T7</b> &nbsp;(medium, 10 min) etl(orders, products): in each category, the customer who received the most delivered items ...</summary>

```python
def etl(orders, products):
    items = (orders.filter((F.col("status") == "delivered") & F.col("customer_id").isNotNull())
                   .join(products, "product_id")
                   .groupBy("category", "customer_id")
                   .agg(F.sum("quantity").alias("items")))
    w = Window.partitionBy("category").orderBy(F.desc("items"), "customer_id")
    return (items.withColumn("rn", F.row_number().over(w))
                 .filter(F.col("rn") == 1)
                 .select("category", "customer_id", "items")
                 .orderBy("category"))

etl(orders, products).show()
```

</details>

<details>
<summary><b>T8</b> &nbsp;(hard, 15 min) etl(orders): for each customer with at least 2 orders, the longest gap in days between two ...</summary>

```python
def etl(orders):
    w = Window.partitionBy("customer_id").orderBy("order_date", "order_id")
    return (orders.filter(F.col("customer_id").isNotNull())
                  .withColumn("gap_days", F.datediff("order_date", F.lag("order_date").over(w)))
                  .groupBy("customer_id")
                  .agg(F.max("gap_days").alias("longest_gap_days"))
                  .filter(F.col("longest_gap_days").isNotNull())
                  .orderBy("customer_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>T9</b> &nbsp;(hard, 15 min) etl(orders): for each customer, the longest run of consecutive months (1, 2, 3) in which they ...</summary>

```python
def etl(orders):
    months = (orders.filter(F.col("customer_id").isNotNull())
                    .select("customer_id", F.month("order_date").alias("month")).distinct())
    w = Window.partitionBy("customer_id").orderBy("month")
    return (months.withColumn("streak_id", F.col("month") - F.row_number().over(w))
                  .groupBy("customer_id", "streak_id").count()
                  .groupBy("customer_id")
                  .agg(F.max("count").alias("longest_streak"))
                  .orderBy("customer_id"))

etl(orders).show()
```

</details>

<details>
<summary><b>T10</b> &nbsp;(hard, 15 min) etl(orders, products): rank the customers by delivered revenue (1 = highest; guests don't ...</summary>

```python
def etl(orders, products):
    revenue = (orders.filter((F.col("status") == "delivered") & F.col("customer_id").isNotNull())
                     .join(products, "product_id")
                     .groupBy("customer_id")
                     .agg(F.sum(F.col("quantity") * F.col("price")).alias("revenue")))
    w = Window.orderBy(F.desc("revenue"))
    return (revenue.withColumn("rank", F.rank().over(w))
                   .withColumn("gap_to_next_better", F.lag("revenue").over(w) - F.col("revenue"))
                   .orderBy("rank"))

etl(orders, products).show()
```

</details>